# Capstone FDE Portfolio Project — Reference Patterns — Hands-On

**FDE Delivery · Week 24a**

Offline notebook: capstone architecture shape, ten deliverables, readiness scorecard, prompt contract regression report, and a portfolio README opening block.

## 0. Capstone architecture shape

```mermaid
flowchart LR
  Problem[Business problem + BLUF] --> Data[Data ingest + ACLs]
  Data --> Core[RAG / agent core]
  Core --> Guard[Guardrails + eval]
  Guard --> API[API + UI]
  API --> Deploy[Deploy + monitor]
  Deploy --> Handoff[Docs, risk, roadmap]
```

The capstone is not just the core model path. It is the business claim, evidence, deployment, monitoring, risk posture, and handoff artifacts around the model path.

In [ ]:
from enum import Enum
from collections import defaultdict
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field
print('Imports ready')

## 1. The 10 canonical deliverables

In [ ]:
deliverables = [
    ('GitHub repo','Proves engineering hygiene','CI passing, src/tests/prompts/evals/infra/docs'),
    ('README','Sells value quickly','BLUF, eval table, run path, cost/query'),
    ('Architecture diagram','Communicates shape','C4 from Mermaid or PlantUML source'),
    ('API documentation','Shows contract','Generated OpenAPI with examples'),
    ('Prompt contract','Treats prompts as code','version, schema, safety rules, changelog'),
    ('Evaluation report','Makes quality credible','golden set, regression, chart, failures'),
    ('Demo video','Shows outcome','3-act arc, not a UI tour'),
    ('Deployment guide','Makes it reproducible','docker compose and cloud infra commands'),
    ('Risk checklist','Earns enterprise trust','threat model, PII/DLP, mitigations'),
    ('Roadmap','Frames what is next','not built, rationale, revisit triggers'),
]
for name, why, signal in deliverables:
    print(f'{name:22s} | {why:28s} | {signal}')

## 2. Capstone deliverable scorecard

In [ ]:
class Status(str, Enum): missing='missing'; stub='stub'; draft='draft'; polished='polished'
class Signal(str, Enum):
    has_eval_numbers='has_eval_numbers'; has_diagram_from_source='has_diagram_from_source'; has_cost_analysis='has_cost_analysis'; has_risk_register='has_risk_register'; has_deployment_guide='has_deployment_guide'; reproducible_from_repo='reproducible_from_repo'; demo_video_present='demo_video_present'; readme_bluf_in_first_paragraph='readme_bluf_in_first_paragraph'; ci_passing='ci_passing'; prompt_contract_versioned='prompt_contract_versioned'; api_docs_generated='api_docs_generated'
EXPECTED = {'repo':{Signal.ci_passing,Signal.reproducible_from_repo}, 'readme':{Signal.readme_bluf_in_first_paragraph,Signal.has_eval_numbers,Signal.has_cost_analysis}, 'architecture':{Signal.has_diagram_from_source}, 'api_docs':{Signal.api_docs_generated,Signal.reproducible_from_repo}, 'prompt_contract':{Signal.prompt_contract_versioned}, 'evaluation':{Signal.has_eval_numbers,Signal.reproducible_from_repo}, 'demo':{Signal.demo_video_present,Signal.reproducible_from_repo,Signal.has_eval_numbers}, 'deployment':{Signal.has_deployment_guide,Signal.reproducible_from_repo}, 'risk':{Signal.has_risk_register}, 'roadmap':{Signal.has_cost_analysis}}
POINTS = {Status.missing:0, Status.stub:25, Status.draft:60, Status.polished:100}
class CapstoneDeliverable(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str; category: str; status: Status; evidence_ref: str; quality_signals: list[Signal] = Field(default_factory=list)
    def score(self):
        exp = EXPECTED.get(self.category, set()); coverage = len(exp & set(self.quality_signals)) / len(exp) if exp else 1
        return int(round(POINTS[self.status] * .7 + coverage * 30))
    def gaps(self):
        gaps = [] if self.status == Status.polished else [f'{self.title}: status is {self.status.value}']
        gaps += [f'{self.title}: missing {m.value}' for m in sorted(EXPECTED.get(self.category,set()) - set(self.quality_signals), key=lambda s: s.value)]
        return gaps
class CapstoneScorecard(BaseModel):
    model_config = ConfigDict(extra='forbid')
    name: str; deliverables: list[CapstoneDeliverable]
    def evaluate(self):
        by_cat = defaultdict(list)
        for d in self.deliverables: by_cat[d.category].append(d.score())
        overall = round(sum(d.score() for d in self.deliverables) / len(self.deliverables), 1)
        gaps = [g for d in self.deliverables for g in d.gaps()]
        verdict = 'portfolio-ready' if overall >= 90 and not gaps else 'demo-ready' if overall >= 75 else 'working-prototype' if overall >= 50 else 'early'
        return {'overall': overall, 'per_category': {k:round(sum(v)/len(v),1) for k,v in sorted(by_cat.items())}, 'top_5_gaps': gaps[:5], 'verdict': verdict}
    def print_report(self):
        r = self.evaluate(); print(f"# {self.name}: {r['overall']} -> {r['verdict']}")
        for k,v in r['per_category'].items(): print(f'{k:16s} {v:5.1f}')
        print('top gaps:', r['top_5_gaps'] or 'none')

In [ ]:
early = CapstoneScorecard(name='Underwriting Policy Assistant early', deliverables=[
 CapstoneDeliverable(title='GitHub repo structure',category='repo',status='polished',evidence_ref='/',quality_signals=[Signal.ci_passing,Signal.reproducible_from_repo]),
 CapstoneDeliverable(title='README with BLUF',category='readme',status='draft',evidence_ref='README.md',quality_signals=[Signal.readme_bluf_in_first_paragraph]),
 CapstoneDeliverable(title='C4 architecture diagram',category='architecture',status='polished',evidence_ref='docs/architecture.md',quality_signals=[Signal.has_diagram_from_source]),
 CapstoneDeliverable(title='OpenAPI docs',category='api_docs',status='draft',evidence_ref='src/api/openapi.json',quality_signals=[Signal.api_docs_generated]),
 CapstoneDeliverable(title='Prompt contract registry',category='prompt_contract',status='stub',evidence_ref='docs/prompt-contract.md',quality_signals=[]),
 CapstoneDeliverable(title='Golden set',category='evaluation',status='draft',evidence_ref='evals/golden.jsonl',quality_signals=[Signal.reproducible_from_repo]),
 CapstoneDeliverable(title='Eval report chart',category='evaluation',status='missing',evidence_ref='docs/evaluation.md',quality_signals=[]),
 CapstoneDeliverable(title='Demo video',category='demo',status='stub',evidence_ref='docs/demo.md',quality_signals=[]),
 CapstoneDeliverable(title='Docker compose',category='deployment',status='draft',evidence_ref='docker-compose.yml',quality_signals=[Signal.has_deployment_guide]),
 CapstoneDeliverable(title='Cloud deploy',category='deployment',status='missing',evidence_ref='infra/',quality_signals=[]),
 CapstoneDeliverable(title='Risk register',category='risk',status='draft',evidence_ref='docs/risk-register.md',quality_signals=[Signal.has_risk_register]),
 CapstoneDeliverable(title='Cost model',category='readme',status='missing',evidence_ref='docs/cost.md',quality_signals=[]),
 CapstoneDeliverable(title='Roadmap',category='roadmap',status='polished',evidence_ref='docs/roadmap.md',quality_signals=[Signal.has_cost_analysis]),
 CapstoneDeliverable(title='CI eval job',category='repo',status='draft',evidence_ref='.github/workflows/eval.yml',quality_signals=[Signal.ci_passing]),
 CapstoneDeliverable(title='Screenshots and GIF',category='demo',status='polished',evidence_ref='docs/assets/',quality_signals=[Signal.demo_video_present]),
])
early.print_report()

In [ ]:
polished = CapstoneScorecard(name='Underwriting Policy Assistant polished', deliverables=[d.model_copy(update={'status': Status.polished, 'quality_signals': sorted(EXPECTED.get(d.category,set()), key=lambda s:s.value)}) for d in early.deliverables])
polished.print_report()

## 3. Prompt contract and eval regression report renderer

In [ ]:
class ChangelogEntry(BaseModel):
    model_config = ConfigDict(extra='forbid')
    version: str; date: str; change_summary: str; delta_grounded_pp: float; delta_cost_pct: float
class PromptContract(BaseModel):
    model_config = ConfigDict(extra='forbid')
    id: str; purpose: str; version: str; model: str; expected_output_schema: dict; safety_rules: list[str]; changelog: list[ChangelogEntry]
    def render_markdown(self):
        lines=[f'# Prompt Contract: {self.id}', f'- Purpose: {self.purpose}', f'- Version: {self.version}', f'- Model: {self.model}', '', '## Schema']
        lines += [f'- `{k}`: {v}' for k,v in self.expected_output_schema.items()]
        lines += ['', '## Safety rules'] + [f'- {r}' for r in self.safety_rules]
        lines += ['', '| Version | Date | Change | Δ grounded pp | Δ cost % |','|---|---|---|---:|---:|']
        lines += [f'| {c.version} | {c.date} | {c.change_summary} | {c.delta_grounded_pp:+.1f} | {c.delta_cost_pct:+.1f}% |' for c in self.changelog]
        return '\n'.join(lines)
class EvalVersion(BaseModel):
    model_config = ConfigDict(extra='forbid')
    version: str; grounded_pct: float; hallucination_rate_pct: float; refusal_rate_pct: float; avg_cost_usd: float; avg_latency_ms: int; verdict: Literal['promote','hold','rollback']
class EvalRegressionReport(BaseModel):
    model_config = ConfigDict(extra='forbid')
    golden_set_size: int; prompt_id: str; versions_compared: list[EvalVersion]
    def render_markdown(self):
        lines=[f'# Eval Regression Report: {self.prompt_id}', f'Golden set size: **{self.golden_set_size}**', '', '| Version | Grounded | Hallucination | Refusal | Cost | Latency | Verdict |','|---|---:|---:|---:|---:|---:|---|']
        for v in self.versions_compared: lines.append(f'| {v.version} | {v.grounded_pct:.1f}% | {v.hallucination_rate_pct:.1f}% | {v.refusal_rate_pct:.1f}% | ${v.avg_cost_usd:.4f} | {v.avg_latency_ms} ms | **{v.verdict}** |')
        return '\n'.join(lines)

In [ ]:
contract = PromptContract(id='underwriter-policy-responder', purpose='Answer underwriter policy questions with cited clauses and refusal on insufficient evidence.', version='v1.4', model='gpt-4o-mini-primary', expected_output_schema={'answer':'string|null','citations':'list[str]','risk_flags':'list[str]','confidence':'0..1','refused':'bool'}, safety_rules=['Use retrieved policy text only as evidence.', 'Refuse without cited support.', 'Never expose PII.', 'Flag stale or ambiguous policy.'], changelog=[ChangelogEntry(version='v1.0',date='2026-06-01',change_summary='Initial policy QA prompt.',delta_grounded_pp=0,delta_cost_pct=0), ChangelogEntry(version='v1.1',date='2026-06-04',change_summary='Added citation-required refusal path.',delta_grounded_pp=5.4,delta_cost_pct=2), ChangelogEntry(version='v1.2',date='2026-06-08',change_summary='Added jurisdiction examples.',delta_grounded_pp=3.1,delta_cost_pct=4.5), ChangelogEntry(version='v1.3',date='2026-06-12',change_summary='Compressed context too aggressively; rolled back.',delta_grounded_pp=-6.8,delta_cost_pct=-18), ChangelogEntry(version='v1.4',date='2026-06-16',change_summary='Restored top-k evidence and stale-policy flag.',delta_grounded_pp=4.9,delta_cost_pct=6)])
report = EvalRegressionReport(golden_set_size=220, prompt_id=contract.id, versions_compared=[EvalVersion(version='v1.0',grounded_pct=82.4,hallucination_rate_pct=6.8,refusal_rate_pct=4.1,avg_cost_usd=.0180,avg_latency_ms=980,verdict='hold'), EvalVersion(version='v1.1',grounded_pct=87.8,hallucination_rate_pct=3.1,refusal_rate_pct=6.5,avg_cost_usd=.0184,avg_latency_ms=1010,verdict='hold'), EvalVersion(version='v1.2',grounded_pct=90.9,hallucination_rate_pct=2.4,refusal_rate_pct=7.0,avg_cost_usd=.0192,avg_latency_ms=1045,verdict='promote'), EvalVersion(version='v1.3',grounded_pct=84.1,hallucination_rate_pct=5.9,refusal_rate_pct=5.2,avg_cost_usd=.0157,avg_latency_ms=890,verdict='rollback'), EvalVersion(version='v1.4',grounded_pct=92.6,hallucination_rate_pct=1.8,refusal_rate_pct=7.4,avg_cost_usd=.0203,avg_latency_ms=1070,verdict='promote')])
print(contract.render_markdown())
print('\n---\n')
print(report.render_markdown())

## 4. Portfolio narrative README opening example

```markdown
# Underwriting Policy Assistant

**BLUF:** This capstone reduces underwriting policy-answer lookup from a 12-minute manual search to a 2.7-minute cited assistant workflow while achieving 92.6% groundedness on a 220-query golden set at $0.020 average cost per answer.

![demo gif](docs/assets/demo.gif) · [3-minute walkthrough](https://example.com/demo-video-placeholder)

| Metric | Result |
|---|---:|
| Golden set size | 220 |
| Groundedness | 92.6% |
| Hallucination rate | 1.8% |
| Refusal rate | 7.4% |
| Avg latency | 1070 ms |
| Avg cost | $0.0203 |

Run locally: `make setup && make test && make eval && make run`.

Security posture: JWT auth, ACL-filtered retrieval, PII-redacted traces, prompt-injection guardrails, risk register, and documented known limitations.
```

## Links
- Literature note: `02 Literature Notes/FDE Delivery/Capstone FDE Portfolio Project — Reference Patterns`
- Snippets: `04 Code Snippets/FDE Delivery/FDE Week 24a Capstone Deliverable Scorecard`, `.../FDE Week 24a Prompt Contract Eval Regression Renderer`
- MOC: `06 Maps of Content/FDE Delivery Concepts`